# Prompt Engineering - czas to zastosować!

Na wykładzie poznaliście podstawy promptowania i few-shot learningu.
Teraz sprawdzimy to w praktyce na trzech zadaniach.

Używamy modelu **Polka-1.1B** (`eryk-mazus/polka-1.1b`) - małego polskiego modelu.
Właśnie dlatego że jest mały, zobaczycie wyraźnie kiedy prompt działa, a kiedy nie.


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="eryk-mazus/polka-1.1b")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

## Zadanie 1: Nadaj modelowi osobowość 🏴‍☠️

Najprostszy trick w promptowaniu — powiedzieć modelowi **kim ma być**.

Wystarczy zacząć prompt od czegoś w stylu:
"Jesteś piratem. Odpowiadasz tylko w stylu pirata."

A potem dać mu jakieś pytanie do odpowiedzi.

Spróbuj kilku różnych person i sprawdź jak zmienia się odpowiedź:
- Pirat 🏴‍☠️
- Astronauta 🚀  
- Średniowieczny rycerz ⚔️
- ... albo cokolwiek wymyślisz

In [ ]:
# Zadanie: wpisz swoją personę!
system_prompt = ...

# Lista pytań które model dostanie po kolei
pytania = [
    "Czym jest internet?",
    "Jak działa samolot?",
    "Co to jest liczba pi?",
]

def przeprowadz_rozmowe(system_prompt, pytania):
    kontekst = system_prompt + "\n\n"

    for pytanie in pytania:
        kontekst += f"Pytanie: {pytanie}\nOdpowiedź:"

        wynik = generator(kontekst, max_new_tokens=60, do_sample=True,
                          temperature=0.8, return_full_text=False)
        odpowiedz = wynik[0]["generated_text"].split("\n")[0]  # bierzemy pierwszą linię

        print(f"❓ {pytanie}")
        print(f"🤖 {odpowiedz}")
        print()

        kontekst += odpowiedz + "\n\n"  # dokładamy do kontekstu na następne pytanie

przeprowadz_rozmowe(system_prompt, pytania)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Czym jest internet?
🤖  Internet to system połączeń sieciowych zawierający różne informacje. Jeśli masz internet, masz dostęp do internetu. Jest to system połączeń sieciowych zawierający różne informacje. Internet to zbiór połączeń sieciowych. Internet jest to system połączeń sieci



Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Jak działa samolot?
🤖  Po połączeniu ze sobą części samolot może podążać w długą drogę. Oznaką, że samolot jest połączony ze sobą częściami jest to, że rozwinął skrzydła. Każdy samolot może wyprodukować własne skrzydła. Ka

❓ Co to jest liczba pi?
🤖  Pi to liczba, która jest ostatecznym wynikiem dzielenia danej liczby przez liczbę równoległą do niej. liczba pi to wynik dzielenia 3 przez zero 3 to liczba równoległa do zero 



## Zadanie 2: Few-shot - naucz model dodawać

Na wykładzie widzieliście, że modelowi można pokazać kilka przykładów
bezpośrednio w prompcie - i on łapie wzorzec.

Sprawdzimy to na dodawaniu. Model językowy **nie jest kalkulatorem**,
ale jeśli pokażemy mu wystarczająco dużo przykładów w odpowiednim formacie...
może zadziała?

Twoje zadanie: dobrać odpowiednią liczbę i format przykładów żeby model
poprawnie uzupełnił wynik.

In [ ]:
przyklady = [
    (2, 3, 5),
    (10, 4, 14),
    # podaj więcej...
]

def zbuduj_prompt(przyklady, a, b):
    prompt = ""
    for x, y, wynik in przyklady:
        prompt += ...  # Zadanie: dodaj linię w formacie "2 + 3 = 5\n"
    prompt += ...      # Zadanie: dodaj nowe zadanie BEZ wyniku na końcu
    return prompt


zadania = [(7, 8), (15, 6), (11, 11), (4, 18), (100, 23)]

for a, b in zadania:
    prompt = zbuduj_prompt(przyklady, a, b)
    wynik = generator(prompt, max_new_tokens=5, do_sample=False)
    odpowiedz = wynik[0]["generated_text"][len(prompt):].strip().split()[0]
    poprawny = a + b
    ok = "✅" if odpowiedz == str(poprawny) else "❌"
    print(f"{ok} {a} + {b} = {odpowiedz}  (powinno być: {poprawny})")

## Zadanie 3: Wyciągnij dane ze zdania 🗂️

Często chcemy wyciągnąć konkretne informacje z tekstu i zapisać je
w ustrukturyzowanym formacie - np. jako słownik Pythona.

Zamiast pisać parsera, możemy po prostu... poprosić model.

Twoim zadaniem jest napisać prompt który wyciągnie z podanego zdania:
- imię i nazwisko
- ulica na której mieszka

i zwróci je jako słownik Pythona `{"imie": ..., "nazwisko": ..., "ulica": ...}`.


In [ ]:
# --- KONFIGURACJA I PRZYKŁADY ---
przyklady_dict = [
    ("Nazywam się Anna Nowak i mieszkam na Polnej 3", "{'imię': 'Anna', 'nazwisko': 'Nowak', 'ulica': 'Polna 3'}"),
    ("Jan Kowalski, adres: Kwiatowa 10", "{'imię': 'Jan', 'nazwisko': 'Kowalski', 'ulica': 'Kwiatowa 10'}"),
]

def zbuduj_prompt(tekst_wejsciowy):
    """Tworzy prompt instruujący model, by zwrócił słownik Pythona."""
    prompt = "..."  # TODO: podaj instrukcje dla modelu

    # TODO: dodaj few-shoty


    return prompt

def napraw_dict(surowy_tekst):
    """
    Czyści tekst tak, aby dało się go zamienić na słownik Pythona (dict).
    Usuwa zbędne komentarze modelu przed i po klamrach.
    """
    # TODO: zostaw tylko pierwszą linię tekstu (małe modele językowe (SLM) potrafią się czasem zapętlić, co zabuża wyniki), możesz użyć w tym celu funkcji .split()
    wynik = ...

    # Logika naprawcza: wyciąganie zawartości między klamrami
    if "{" in wynik and "}" in wynik:  # TODO: wyciągnij obszar gdzie jest dict
        ...
    elif "{" in wynik and "}" not in wynik:  # TODO: jeśli model nie skończył generować to trzeba to też obsłużyć
        ...

    return wynik

# --- TESTY I SPRAWDZACZKA ---
testy = [
    "Cześć, tu Marek Wójcik, wysyłka na Leśną 12",
    "Paczka dla: Kasia Balcerzak, ulica Długa 5",
    "Dzień dobry, nazywam się Robert Lewandowski. Mieszkam przy ul. Głównej 1"
]

print("🚀 Uruchamiam Ekstraktor Danych...\n")

for tekst in testy:
    full_prompt = zbuduj_prompt(tekst)

    output = ... # TODO: użyj naszego generatora

    # Próba naprawy i konwersji na obiekt Pythona
    oczyszczony_tekst = napraw_dict(output)

    print(f"📥 Wejście: {tekst}")
    print(f"🤖 Model wypluł: {output}")

    try:
        dane = eval(oczyszczony_tekst)

        if isinstance(dane, dict):
            wymagane = ["imię", "nazwisko", "ulica"]
            brakujace = [k for k in wymagane if k not in dane]

            if brakujace:
                print(f"⚠️  Słownik stworzony, ale brakuje pól: {', '.join(brakujace)}")
            else:
                print(f"✅ Sukces! Dane w Pythonie: {dane['imię']} {dane['nazwisko']} -> {dane['ulica']}")
        else:
            print("❌ Model nie zwrócił słownika.")

    except (SyntaxError, ValueError):
        print(f"❌ Błąd składni: Nie udało się sparsować tekstu na słownik.")
        print(f"   Próba naprawy: {oczyszczony_tekst}")

    print("-" * 50, "\n")

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 Uruchamiam Ekstraktor Danych (Wersja: Python Dict)...



Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Wejście: Cześć, tu Marek Wójcik, wysyłka na Leśną 12
🤖 Model wypluł:  {'imię': 'Marek', 'nazwisko': 'Wójcik', 'ulica': 'Leśna 12'}
✅ Sukces! Dane w Pythonie: Marek Wójcik -> Leśna 12
-------------------------------------------------- 



Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📥 Wejście: Paczka dla: Kasia Balcerzak, ulica Długa 5
🤖 Model wypluł:  {'imię': 'Kasia', 'nazwisko': 'Balcerzak', 'ulica': 'Długa 5'}
✅ Sukces! Dane w Pythonie: Kasia Balcerzak -> Długa 5
-------------------------------------------------- 

📥 Wejście: Dzień dobry, nazywam się Robert Lewandowski. Mieszkam przy ul. Głównej 1
🤖 Model wypluł:  {'imię': 'Robert', 'nazwisko': 'Lewandowski', 'ulica': 'Główna 1'}
✅ Sukces! Dane w Pythonie: Robert Lewandowski -> Główna 1
-------------------------------------------------- 

